# 08 — Historical Signal Backtest

This notebook evaluates whether the combined MarketGuard signals produced meaningful historical separation.

The backtest will:

- Use only out-of-sample dates from 2024 onward
- Generate historical outperform and downside scores
- Rank stocks cross-sectionally on each rebalance date
- Recreate opportunity tiers and downside-risk classes
- Compare future returns, Nifty 50 excess returns, drawdowns, and downside-event rates
- Exclude limited-history stocks from normal-confidence rankings

The analysis evaluates model signals as a screening system, not as an executable trading strategy. Transaction costs, turnover, and portfolio construction will be handled separately.

### Load data and saved models

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    """Find the MarketGuard repository root."""
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "config.yaml").exists() or (path / "README.md").exists():
            return path

    raise FileNotFoundError(
        "Could not find the project root containing config.yaml or README.md."
    )


def extract_feature_names(model) -> list[str]:
    """Extract the exact training feature order from a saved sklearn pipeline."""
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Feature names were not stored in the saved model pipeline."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

TARGET_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

MODEL_DIR = PROJECT_ROOT / "models"

OUTPERFORM_MODEL_PATH = (
    MODEL_DIR
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    MODEL_DIR
    / "random_forest_downside_10pct_20d_v1.joblib"
)

BACKTEST_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "historical_signal_backtest"
)

BACKTEST_REPORT_DIR.mkdir(parents=True, exist_ok=True)

BACKTEST_START_DATE = pd.Timestamp("2024-01-01")


required_paths = [
    TARGET_DATA_PATH,
    OUTPERFORM_MODEL_PATH,
    DOWNSIDE_MODEL_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)

    raise FileNotFoundError(
        f"Required backtest assets are missing:\n{missing_text}"
    )


# Load historical feature and target dataset
data = pd.read_parquet(TARGET_DATA_PATH)

data["date"] = pd.to_datetime(data["date"])

data = (
    data
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)


# Load saved models
outperform_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_model = joblib.load(DOWNSIDE_MODEL_PATH)

outperform_feature_cols = extract_feature_names(outperform_model)
downside_feature_cols = extract_feature_names(downside_model)

if outperform_feature_cols != downside_feature_cols:
    raise ValueError(
        "The two saved models use different feature orders."
    )

feature_cols = outperform_feature_cols


missing_feature_cols = [
    column
    for column in feature_cols
    if column not in data.columns
]

if missing_feature_cols:
    raise ValueError(
        f"Dataset is missing model features: {missing_feature_cols}"
    )


# Inspect available future-outcome and target columns
candidate_target_cols = [
    column
    for column in data.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "future",
            "target_",
            "outperform",
            "downside",
        ]
    )
]


print("Project root:")
print(PROJECT_ROOT)

print("\nDataset shape:")
print(data.shape)

print("\nDataset date range:")
print(data["date"].min(), "to", data["date"].max())

print("\nBacktest start date:")
print(BACKTEST_START_DATE)

print("\nUnique stocks:")
print(data["yf_ticker"].nunique())

print("\nFeature count:")
print(len(feature_cols))

print("\nBoth models use identical feature order:")
print(outperform_feature_cols == downside_feature_cols)

print("\nPotential historical outcome columns:")
for column in candidate_target_cols:
    print(column)

print("\nBacktest report directory:")
print(BACKTEST_REPORT_DIR)

Project root:
E:\Projects\marketguard-india

Dataset shape:
(357370, 200)

Dataset date range:
2010-01-04 00:00:00 to 2026-07-14 00:00:00

Backtest start date:
2024-01-01 00:00:00

Unique stocks:
100

Feature count:
79

Both models use identical feature order:
True

Potential historical outcome columns:
future_adj_close_5d
future_return_5d
target_available_5d
future_adj_close_20d
future_return_20d
target_available_20d
future_adj_close_60d
future_return_60d
target_available_60d
future_min_adj_close_5d
future_max_adj_close_5d
future_min_return_5d
future_max_return_5d
path_target_available_5d
future_min_adj_close_20d
future_max_adj_close_20d
future_min_return_20d
future_max_return_20d
path_target_available_20d
future_min_adj_close_60d
future_max_adj_close_60d
future_min_return_60d
future_max_return_60d
path_target_available_60d
target_ready_v1_20d
target_ready_v1_clean_20d
target_positive_return_20d
target_downside_5pct_20d
target_big_downside_10pct_20d
target_upside_5pct_20d
target_retur

We now have the exact outcomes needed. The next step is to build a clean out-of-sample monthly backtest sample.

We use monthly rebalance dates because the prediction horizon is 20 trading days. Testing every day would create heavily overlapping outcomes and overstate the amount of independent evidence.

### Prepare monthly out-of-sample backtest rows

In [2]:
# Historical outcomes used in the backtest

required_outcome_cols = [
    "future_return_20d",
    "future_nifty50_return_20d",
    "future_excess_return_20d_vs_nifty50",
    "future_min_return_20d",
    "future_max_return_20d",
    "target_outperform_nifty50_20d",
    "target_downside_5pct_20d",
    "target_big_downside_10pct_20d",
]

missing_outcome_cols = [
    column
    for column in required_outcome_cols
    if column not in data.columns
]

if missing_outcome_cols:
    raise ValueError(
        f"Required historical outcomes are missing: {missing_outcome_cols}"
    )


# Use only rows after the model-training period
backtest_base = data.loc[
    data["date"] >= BACKTEST_START_DATE
].copy()


# Apply the same stock-readiness rules used during modeling
eligibility_mask = pd.Series(
    True,
    index=backtest_base.index,
)

if "model_ready_v1" in backtest_base.columns:
    eligibility_mask &= (
        backtest_base["model_ready_v1"]
        .fillna(False)
        .astype(bool)
    )

if "target_ready_v1_clean_20d" in backtest_base.columns:
    eligibility_mask &= (
        backtest_base["target_ready_v1_clean_20d"]
        .fillna(False)
        .astype(bool)
    )


# Require complete future outcomes for evaluation
eligibility_mask &= (
    backtest_base[required_outcome_cols]
    .notna()
    .all(axis=1)
)


eligible_backtest_rows = (
    backtest_base.loc[eligibility_mask]
    .copy()
)


# Select the first available market date in each calendar month
available_dates = (
    eligible_backtest_rows[["date"]]
    .drop_duplicates()
    .sort_values("date")
    .copy()
)

available_dates["rebalance_month"] = (
    available_dates["date"]
    .dt.to_period("M")
)

monthly_rebalance_dates = (
    available_dates
    .groupby("rebalance_month", as_index=False)
    .first()
    .rename(columns={"date": "rebalance_date"})
)


# Keep only monthly rebalance observations
monthly_backtest = (
    eligible_backtest_rows.loc[
        eligible_backtest_rows["date"].isin(
            monthly_rebalance_dates["rebalance_date"]
        )
    ]
    .copy()
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)


# Check feature quality before scoring
monthly_features = (
    monthly_backtest[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
)

monthly_backtest["missing_model_feature_count"] = (
    monthly_features
    .isna()
    .sum(axis=1)
)


rebalance_summary = (
    monthly_backtest
    .groupby("date")
    .agg(
        stock_count=("yf_ticker", "nunique"),
        missing_feature_rows=(
            "missing_model_feature_count",
            lambda values: int((values > 0).sum()),
        ),
    )
    .reset_index()
)


print("Eligible daily out-of-sample rows:")
print(len(eligible_backtest_rows))

print("\nMonthly rebalance dates:")
print(monthly_backtest["date"].nunique())

print("\nMonthly backtest rows:")
print(len(monthly_backtest))

print("\nBacktest date range:")
print(
    monthly_backtest["date"].min(),
    "to",
    monthly_backtest["date"].max(),
)

print("\nStocks per rebalance date:")
display(
    rebalance_summary[
        ["stock_count", "missing_feature_rows"]
    ]
    .describe()
)

print("\nFirst five rebalance dates:")
display(rebalance_summary.head())

print("\nLast five rebalance dates:")
display(rebalance_summary.tail())

print("\nMissing-feature distribution:")
display(
    monthly_backtest["missing_model_feature_count"]
    .value_counts()
    .sort_index()
    .rename_axis("missing_feature_count")
    .reset_index(name="row_count")
)

Eligible daily out-of-sample rows:
53640

Monthly rebalance dates:
30

Monthly backtest rows:
2613

Backtest date range:
2024-01-02 00:00:00 to 2026-06-01 00:00:00

Stocks per rebalance date:


,stock_count,missing_feature_rows
count,30.000000,30.0
mean,87.100000,0.0
std,15.883954,0.0
min,3.000000,0.0
25%,90.000000,0.0
50%,90.000000,0.0
75%,90.000000,0.0
max,90.000000,0.0



First five rebalance dates:


,date,stock_count,missing_feature_rows
0,2024-01-02,90,0
1,2024-02-01,90,0
2,2024-03-01,90,0
3,2024-04-01,90,0
4,2024-05-02,90,0



Last five rebalance dates:


,date,stock_count,missing_feature_rows
25,2026-02-02,90,0
26,2026-03-02,90,0
27,2026-04-01,3,0
28,2026-05-04,90,0
29,2026-06-01,90,0



Missing-feature distribution:


,missing_feature_count,row_count
0,0,2613


The backtest sample is mostly correct, but 2026-04-01 having only 3 stocks is a red flag. Every other full rebalance date has 90 stocks, so we should diagnose this before generating model scores.

The missing-feature result is good: all 2,613 selected rows have complete model inputs.

### Investigate the April 2026 anomaly

In [3]:
diagnostic_date = pd.Timestamp("2026-04-01")

diagnostic_rows = (
    backtest_base.loc[
        backtest_base["date"] == diagnostic_date
    ]
    .copy()
)

diagnostic_rows["model_ready_check"] = (
    diagnostic_rows["model_ready_v1"]
    .fillna(False)
    .astype(bool)
)

diagnostic_rows["target_ready_check"] = (
    diagnostic_rows["target_ready_v1_clean_20d"]
    .fillna(False)
    .astype(bool)
)

diagnostic_rows["all_outcomes_available"] = (
    diagnostic_rows[required_outcome_cols]
    .notna()
    .all(axis=1)
)

diagnostic_rows["final_eligible_check"] = (
    diagnostic_rows["model_ready_check"]
    & diagnostic_rows["target_ready_check"]
    & diagnostic_rows["all_outcomes_available"]
)


diagnostic_summary = pd.DataFrame({
    "check": [
        "Total rows",
        "Unique stocks",
        "Model ready",
        "Target ready",
        "All outcomes available",
        "Final eligible",
    ],
    "stock_count": [
        len(diagnostic_rows),
        diagnostic_rows["yf_ticker"].nunique(),
        int(diagnostic_rows["model_ready_check"].sum()),
        int(diagnostic_rows["target_ready_check"].sum()),
        int(diagnostic_rows["all_outcomes_available"].sum()),
        int(diagnostic_rows["final_eligible_check"].sum()),
    ],
})

print("Eligibility summary for:")
print(diagnostic_date)

display(diagnostic_summary)


print("\nMissing values by outcome column:")

display(
    diagnostic_rows[required_outcome_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "outcome_column"})
)


excluded_display_cols = [
    column
    for column in [
        "symbol",
        "company_name",
        "yf_ticker",
        "model_ready_v1",
        "target_ready_v1_clean_20d",
        "model_ready_check",
        "target_ready_check",
        "all_outcomes_available",
        "final_eligible_check",
    ]
    if column in diagnostic_rows.columns
]

print("\nExcluded stocks:")

display(
    diagnostic_rows.loc[
        ~diagnostic_rows["final_eligible_check"],
        excluded_display_cols,
    ]
    .sort_values("yf_ticker")
)

Eligibility summary for:
2026-04-01 00:00:00


,check,stock_count
0,Total rows,100
1,Unique stocks,100
2,Model ready,90
3,Target ready,90
4,All outcomes available,3
5,Final eligible,3



Missing values by outcome column:


,outcome_column,missing_count
0,future_return_20d,0
1,future_nifty50_return_20d,97
2,future_excess_return_20d_vs_nifty50,97
3,future_min_return_20d,0
4,future_max_return_20d,0
5,target_outperform_nifty50_20d,10
6,target_downside_5pct_20d,0
7,target_big_downside_10pct_20d,0



Excluded stocks:


,symbol,company_name,yf_ticker,model_ready_v1,target_ready_v1_clean_20d,model_ready_check,target_ready_check,all_outcomes_available,final_eligible_check
350073,ABB,ABB India Ltd.,ABB.NS,1,1,True,True,False,False
350074,ADANIENSOL,Adani Energy Solutions Ltd.,ADANIENSOL.NS,1,1,True,True,False,False
350075,ADANIENT,Adani Enterprises Ltd.,ADANIENT.NS,1,1,True,True,False,False
350076,ADANIGREEN,Adani Green Energy Ltd.,ADANIGREEN.NS,1,1,True,True,False,False
350077,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS.NS,1,1,True,True,False,False
...,...,...,...,...,...,...,...,...,...
350168,UNITDSPR,United Spirits Ltd.,UNITDSPR.NS,1,1,True,True,False,False
350169,VBL,Varun Beverages Ltd.,VBL.NS,1,1,True,True,False,False
350170,VEDL,Vedanta Ltd.,VEDL.NS,1,1,True,True,False,False
350171,WIPRO,Wipro Ltd.,WIPRO.NS,1,1,True,True,False,False


The issue is isolated to the benchmark outcome columns:

future_nifty50_return_20d
future_excess_return_20d_vs_nifty50

They are missing for 97 stocks on 2026-04-01, so that date should not be used as a rebalance date. Rather than manually deleting it, select the first date each month with sufficient cross-sectional coverage.
### Rebuild monthly sample using minimum coverage

In [4]:
MIN_STOCKS_PER_REBALANCE = 80

# Count eligible stocks on every available date
daily_coverage = (
    eligible_backtest_rows
    .groupby("date")
    .agg(
        stock_count=("yf_ticker", "nunique"),
    )
    .reset_index()
    .sort_values("date")
)

daily_coverage["rebalance_month"] = (
    daily_coverage["date"].dt.to_period("M")
)

# Keep only dates with enough stocks for meaningful cross-sectional ranking
valid_rebalance_candidates = (
    daily_coverage.loc[
        daily_coverage["stock_count"] >= MIN_STOCKS_PER_REBALANCE
    ]
    .copy()
)

# Select the first sufficiently complete date in each month
monthly_rebalance_dates = (
    valid_rebalance_candidates
    .groupby("rebalance_month", as_index=False)
    .first()
    .rename(columns={"date": "rebalance_date"})
)

monthly_backtest = (
    eligible_backtest_rows.loc[
        eligible_backtest_rows["date"].isin(
            monthly_rebalance_dates["rebalance_date"]
        )
    ]
    .copy()
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)

rebalance_summary = (
    monthly_backtest
    .groupby("date")
    .agg(
        stock_count=("yf_ticker", "nunique"),
    )
    .reset_index()
)

print("Minimum stocks required per rebalance date:")
print(MIN_STOCKS_PER_REBALANCE)

print("\nMonthly rebalance dates:")
print(monthly_backtest["date"].nunique())

print("\nMonthly backtest rows:")
print(len(monthly_backtest))

print("\nBacktest date range:")
print(
    monthly_backtest["date"].min(),
    "to",
    monthly_backtest["date"].max(),
)

print("\nStocks per rebalance date:")
display(rebalance_summary["stock_count"].describe())

print("\nApril 2026 selected rebalance date:")
display(
    rebalance_summary.loc[
        rebalance_summary["date"].dt.to_period("M")
        == pd.Period("2026-04")
    ]
)

print("\nRebalance dates with fewer than 90 stocks:")
display(
    rebalance_summary.loc[
        rebalance_summary["stock_count"] < 90
    ]
)

Minimum stocks required per rebalance date:
80

Monthly rebalance dates:
30

Monthly backtest rows:
2700

Backtest date range:
2024-01-02 00:00:00 to 2026-06-01 00:00:00

Stocks per rebalance date:


count    30.0
mean     90.0
std       0.0
min      90.0
25%      90.0
50%      90.0
75%      90.0
max      90.0
Name: stock_count, dtype: float64


April 2026 selected rebalance date:


,date,stock_count
27,2026-04-02,90



Rebalance dates with fewer than 90 stocks:


,date,stock_count


The monthly sample is now clean:

30 rebalance dates
90 stocks per date
2,700 historical observations
April 2026 correctly moved to 2026-04-02
No missing model features

Next, generate historical model scores and recreate the same classifications used in Notebook 07.

### Score and classify historical observations

In [5]:
# Generate historical model scores

X_backtest = (
    monthly_backtest[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
)

monthly_backtest["outperform_probability"] = (
    outperform_model.predict_proba(X_backtest)[:, 1]
)

monthly_backtest["downside_probability"] = (
    downside_model.predict_proba(X_backtest)[:, 1]
)


# Cross-sectional ranks within each rebalance date
monthly_backtest["outperform_rank"] = (
    monthly_backtest
    .groupby("date")["outperform_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

monthly_backtest["downside_risk_rank"] = (
    monthly_backtest
    .groupby("date")["downside_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

monthly_backtest["rebalance_stock_count"] = (
    monthly_backtest
    .groupby("date")["yf_ticker"]
    .transform("nunique")
)

monthly_backtest["outperform_rank_percentile"] = (
    monthly_backtest["outperform_rank"]
    / monthly_backtest["rebalance_stock_count"]
)


# Recreate opportunity tiers
monthly_backtest["opportunity_tier"] = np.select(
    [
        monthly_backtest["outperform_rank_percentile"] <= 0.20,
        monthly_backtest["outperform_rank_percentile"] <= 0.50,
    ],
    [
        "High Opportunity",
        "Moderate Opportunity",
    ],
    default="Low Opportunity",
)


# Recreate downside-risk bands
monthly_backtest["downside_risk_band"] = pd.cut(
    monthly_backtest["downside_probability"],
    bins=[-np.inf, 0.40, 0.47, 0.51, np.inf],
    labels=[
        "Low Risk",
        "Watch Risk",
        "High Risk",
        "Very High Risk",
    ],
    right=False,
).astype(str)


def assign_combined_class(row) -> str:
    opportunity = row["opportunity_tier"]
    risk = row["downside_risk_band"]

    lower_risk = risk in ["Low Risk", "Watch Risk"]
    higher_risk = risk in ["High Risk", "Very High Risk"]

    if opportunity == "High Opportunity" and lower_risk:
        return "Attractive Risk-Reward"

    if opportunity == "High Opportunity" and higher_risk:
        return "High Opportunity / High Risk"

    if opportunity == "Moderate Opportunity" and lower_risk:
        return "Balanced Opportunity"

    if opportunity == "Moderate Opportunity" and higher_risk:
        return "Caution"

    if opportunity == "Low Opportunity" and lower_risk:
        return "Low Opportunity / Lower Risk"

    return "Unfavourable Risk-Reward"


monthly_backtest["opportunity_risk_class"] = (
    monthly_backtest.apply(
        assign_combined_class,
        axis=1,
    )
)


classification_summary = (
    monthly_backtest
    .groupby("opportunity_risk_class")
    .agg(
        total_observations=("yf_ticker", "size"),
        average_stocks_per_month=(
            "yf_ticker",
            lambda values: len(values) / monthly_backtest["date"].nunique(),
        ),
    )
    .reset_index()
    .sort_values("total_observations", ascending=False)
)


print("Historical observations scored:")
print(len(monthly_backtest))

print("\nRebalance dates:")
print(monthly_backtest["date"].nunique())

print("\nOpportunity-tier counts:")
display(
    monthly_backtest["opportunity_tier"]
    .value_counts()
    .rename_axis("opportunity_tier")
    .reset_index(name="observation_count")
)

print("\nDownside-risk band counts:")
display(
    monthly_backtest["downside_risk_band"]
    .value_counts()
    .rename_axis("downside_risk_band")
    .reset_index(name="observation_count")
)

print("\nCombined classification summary:")
display(classification_summary)

print("\nFirst rebalance-date sample:")
display(
    monthly_backtest.loc[
        monthly_backtest["date"]
        == monthly_backtest["date"].min(),
        [
            "date",
            "symbol",
            "outperform_probability",
            "outperform_rank",
            "opportunity_tier",
            "downside_probability",
            "downside_risk_rank",
            "downside_risk_band",
            "opportunity_risk_class",
        ],
    ]
    .sort_values("outperform_rank")
    .head(15)
)

Historical observations scored:
2700

Rebalance dates:
30

Opportunity-tier counts:


,opportunity_tier,observation_count
0,Low Opportunity,1350
1,Moderate Opportunity,810
2,High Opportunity,540



Downside-risk band counts:


,downside_risk_band,observation_count
0,Low Risk,1292
1,Very High Risk,573
2,Watch Risk,512
3,High Risk,323



Combined classification summary:


,opportunity_risk_class,total_observations,average_stocks_per_month
4,Low Opportunity / Lower Risk,863,28.766667
1,Balanced Opportunity,562,18.733333
5,Unfavourable Risk-Reward,487,16.233333
0,Attractive Risk-Reward,379,12.633333
2,Caution,248,8.266667
3,High Opportunity / High Risk,161,5.366667



First rebalance-date sample:


,date,symbol,outperform_probability,outperform_rank,opportunity_tier,downside_probability,downside_risk_rank,downside_risk_band,opportunity_risk_class
31,2024-01-02,GAIL,0.621892,1,High Opportunity,0.445683,10,Watch Risk,Attractive Risk-Reward
74,2024-01-02,TATAPOWER,0.620694,2,High Opportunity,0.449700,9,Watch Risk,Attractive Risk-Reward
2,2024-01-02,ADANIENT,0.611598,3,High Opportunity,0.542134,5,Very High Risk,High Opportunity / High Risk
5,2024-01-02,ADANIPOWER,0.611447,4,High Opportunity,0.510191,6,Very High Risk,High Opportunity / High Risk
46,2024-01-02,IOC,0.611142,5,High Opportunity,0.415380,14,Watch Risk,Attractive Risk-Reward
3,2024-01-02,ADANIGREEN,0.609026,6,High Opportunity,0.605009,1,Very High Risk,High Opportunity / High Risk
1,2024-01-02,ADANIENSOL,0.606919,7,High Opportunity,0.566338,3,Very High Risk,High Opportunity / High Risk
39,2024-01-02,HINDALCO,0.606305,8,High Opportunity,0.327281,33,Low Risk,Attractive Risk-Reward
58,2024-01-02,NTPC,0.606290,9,High Opportunity,0.338832,25,Low Risk,Attractive Risk-Reward
24,2024-01-02,COALINDIA,0.606275,10,High Opportunity,0.419034,13,Watch Risk,Attractive Risk-Reward


The historical classifications were generated correctly. The category sizes vary by month because the downside scores change with market conditions.

Next, measure how each class actually performed. We will calculate an equal-weight portfolio for every class on every rebalance date, then average those monthly results so months with more stocks do not dominate.

### Evaluate future outcomes by classification

In [6]:
# Evaluate each opportunity-risk class using realised 20-day outcomes

class_order = [
    "Attractive Risk-Reward",
    "High Opportunity / High Risk",
    "Balanced Opportunity",
    "Caution",
    "Low Opportunity / Lower Risk",
    "Unfavourable Risk-Reward",
]


# Equal-weight class outcome for each rebalance date
monthly_class_performance = (
    monthly_backtest
    .groupby(
        ["date", "opportunity_risk_class"],
        observed=True,
    )
    .agg(
        stock_count=("yf_ticker", "size"),
        average_future_return_20d=("future_return_20d", "mean"),
        median_future_return_20d=("future_return_20d", "median"),
        average_excess_return_20d=(
            "future_excess_return_20d_vs_nifty50",
            "mean",
        ),
        average_future_min_return_20d=(
            "future_min_return_20d",
            "mean",
        ),
        outperform_rate=(
            "target_outperform_nifty50_20d",
            "mean",
        ),
        downside_5pct_rate=(
            "target_downside_5pct_20d",
            "mean",
        ),
        downside_10pct_rate=(
            "target_big_downside_10pct_20d",
            "mean",
        ),
    )
    .reset_index()
)


# Aggregate monthly class portfolios with equal weight per month
class_backtest_summary = (
    monthly_class_performance
    .groupby(
        "opportunity_risk_class",
        observed=True,
    )
    .agg(
        months_present=("date", "nunique"),
        total_stock_observations=("stock_count", "sum"),
        average_stocks_per_month=("stock_count", "mean"),

        average_future_return_20d=(
            "average_future_return_20d",
            "mean",
        ),
        median_monthly_future_return_20d=(
            "average_future_return_20d",
            "median",
        ),
        return_volatility_across_months=(
            "average_future_return_20d",
            "std",
        ),

        average_excess_return_20d=(
            "average_excess_return_20d",
            "mean",
        ),
        positive_excess_month_rate=(
            "average_excess_return_20d",
            lambda values: (values > 0).mean(),
        ),

        average_future_min_return_20d=(
            "average_future_min_return_20d",
            "mean",
        ),
        outperform_rate=("outperform_rate", "mean"),
        downside_5pct_rate=("downside_5pct_rate", "mean"),
        downside_10pct_rate=("downside_10pct_rate", "mean"),
    )
    .reindex(class_order)
    .reset_index()
)


# Convert return and event-rate columns to percentages for display
percentage_columns = [
    "average_future_return_20d",
    "median_monthly_future_return_20d",
    "return_volatility_across_months",
    "average_excess_return_20d",
    "positive_excess_month_rate",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

class_backtest_summary_display = class_backtest_summary.copy()

class_backtest_summary_display[percentage_columns] = (
    class_backtest_summary_display[percentage_columns] * 100
)

class_backtest_summary_display = (
    class_backtest_summary_display.round(2)
)


print("Historical class performance:")
display(class_backtest_summary_display)


print("\nMonthly class-performance rows:")
print(len(monthly_class_performance))

print("\nMonths covered by each class:")
display(
    class_backtest_summary_display[
        [
            "opportunity_risk_class",
            "months_present",
            "average_stocks_per_month",
        ]
    ]
)

Historical class performance:


,opportunity_risk_class,months_present,total_stock_observations,average_stocks_per_month,average_future_return_20d,median_monthly_future_return_20d,return_volatility_across_months,average_excess_return_20d,positive_excess_month_rate,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Attractive Risk-Reward,30,379,12.63,2.02,2.39,5.18,1.83,63.33,-3.62,57.31,28.45,7.86
1,High Opportunity / High Risk,22,161,7.32,3.14,2.69,10.06,2.75,54.55,-5.11,56.04,43.61,23.52
2,Balanced Opportunity,29,562,19.38,1.23,0.60,3.57,0.74,58.62,-3.70,52.22,33.16,9.12
3,Caution,25,248,9.92,1.80,1.80,7.15,1.46,72.00,-5.73,52.84,50.96,24.79
4,Low Opportunity / Lower Risk,29,863,29.76,0.42,0.79,3.47,-0.08,48.28,-4.31,47.14,35.76,10.53
5,Unfavourable Risk-Reward,29,487,16.79,0.51,0.78,6.35,0.32,41.38,-5.96,49.86,49.68,19.73



Monthly class-performance rows:
164

Months covered by each class:


,opportunity_risk_class,months_present,average_stocks_per_month
0,Attractive Risk-Reward,30,12.63
1,High Opportunity / High Risk,22,7.32
2,Balanced Opportunity,29,19.38
3,Caution,25,9.92
4,Low Opportunity / Lower Risk,29,29.76
5,Unfavourable Risk-Reward,29,16.79


These results are promising and the combined labels show meaningful separation:

    Attractive Risk-Reward produced solid excess returns (+1.83%) with the lowest 10% downside-event rate (7.86%).

    High Opportunity / High Risk produced the highest return, but with much higher volatility and a 23.52% severe-downside rate.

    Balanced Opportunity delivered moderate positive excess return with relatively controlled downside.

    Caution earned positive returns, but its 24.79% severe-downside rate confirms the risk warning.

    Unfavourable Risk-Reward had weak consistency: only 41.38% of months had positive excess return, with poor drawdowns.

    The positive average excess return for Unfavourable stocks may be caused by a few unusually strong months, so average return alone is not enough.

However, 2024 was used as the validation period during model selection. Therefore, the important next check is whether these results remain present from 2025 onward, which is the cleaner test period.

### Compare validation and test-period performance

In [7]:
# Compare 2024 validation performance against the cleaner 2025+ test period

monthly_class_performance["evaluation_period"] = np.where(
    monthly_class_performance["date"] < pd.Timestamp("2025-01-01"),
    "2024 Validation",
    "2025+ Test",
)

period_class_summary = (
    monthly_class_performance
    .groupby(
        ["evaluation_period", "opportunity_risk_class"],
        observed=True,
    )
    .agg(
        months_present=("date", "nunique"),
        average_stocks_per_month=("stock_count", "mean"),
        average_future_return_20d=(
            "average_future_return_20d",
            "mean",
        ),
        average_excess_return_20d=(
            "average_excess_return_20d",
            "mean",
        ),
        positive_excess_month_rate=(
            "average_excess_return_20d",
            lambda values: (values > 0).mean(),
        ),
        average_future_min_return_20d=(
            "average_future_min_return_20d",
            "mean",
        ),
        outperform_rate=("outperform_rate", "mean"),
        downside_5pct_rate=("downside_5pct_rate", "mean"),
        downside_10pct_rate=("downside_10pct_rate", "mean"),
    )
    .reset_index()
)

period_percentage_columns = [
    "average_future_return_20d",
    "average_excess_return_20d",
    "positive_excess_month_rate",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

period_class_summary_display = period_class_summary.copy()

period_class_summary_display[period_percentage_columns] = (
    period_class_summary_display[period_percentage_columns] * 100
)

period_class_summary_display = (
    period_class_summary_display
    .sort_values(
        ["evaluation_period", "opportunity_risk_class"]
    )
    .round(2)
)

print("Performance by evaluation period:")
display(period_class_summary_display)


# Attractive versus Unfavourable spread
comparison_classes = period_class_summary.loc[
    period_class_summary["opportunity_risk_class"].isin(
        [
            "Attractive Risk-Reward",
            "Unfavourable Risk-Reward",
        ]
    )
].copy()

spread_metrics = [
    "average_future_return_20d",
    "average_excess_return_20d",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

spread_rows = []

for period in comparison_classes["evaluation_period"].unique():
    period_data = (
        comparison_classes.loc[
            comparison_classes["evaluation_period"] == period
        ]
        .set_index("opportunity_risk_class")
    )

    if {
        "Attractive Risk-Reward",
        "Unfavourable Risk-Reward",
    }.issubset(period_data.index):

        row = {"evaluation_period": period}

        for metric in spread_metrics:
            row[f"{metric}_spread"] = (
                period_data.loc[
                    "Attractive Risk-Reward",
                    metric,
                ]
                - period_data.loc[
                    "Unfavourable Risk-Reward",
                    metric,
                ]
            ) * 100

        spread_rows.append(row)

attractive_vs_unfavourable_spread = (
    pd.DataFrame(spread_rows)
    .round(2)
)

print("\nAttractive minus Unfavourable spread:")
display(attractive_vs_unfavourable_spread)

Performance by evaluation period:


,evaluation_period,opportunity_risk_class,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,positive_excess_month_rate,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,2024 Validation,Attractive Risk-Reward,12,13.92,3.63,3.28,91.67,-4.15,63.28,31.61,9.39
1,2024 Validation,Balanced Opportunity,12,20.08,1.41,1.06,66.67,-4.60,51.88,41.99,8.93
2,2024 Validation,Caution,12,6.92,1.12,0.77,58.33,-7.08,46.05,60.97,27.98
3,2024 Validation,High Opportunity / High Risk,10,4.90,4.59,4.47,60.00,-5.27,57.9,45.94,29.48
4,2024 Validation,Low Opportunity / Lower Risk,12,29.75,0.68,0.33,66.67,-4.80,46.26,42.05,10.8
5,2024 Validation,Unfavourable Risk-Reward,11,16.64,-0.28,-0.65,36.36,-7.32,50.87,59.71,28.51
6,2025+ Test,Attractive Risk-Reward,18,11.78,0.94,0.86,44.44,-3.27,53.34,26.35,6.84
7,2025+ Test,Balanced Opportunity,17,18.88,1.11,0.51,52.94,-3.07,52.45,26.92,9.25
8,2025+ Test,Caution,13,12.69,2.43,2.09,84.62,-4.48,59.11,41.73,21.85
9,2025+ Test,High Opportunity / High Risk,12,9.33,1.93,1.32,50.00,-4.98,54.48,41.68,18.55



Attractive minus Unfavourable spread:


,evaluation_period,average_future_return_20d_spread,average_excess_return_20d_spread,average_future_min_return_20d_spread,outperform_rate_spread,downside_5pct_rate_spread,downside_10pct_rate_spread
0,2024 Validation,3.91,3.93,3.17,12.41,-28.1,-19.12
1,2025+ Test,-0.05,-0.05,1.86,4.10,-17.2,-7.53


The cleaner 2025+ test period does not confirm a return advantage for Attractive Risk-Reward:

    Attractive excess return: +0.86%
    Unfavourable excess return: +0.91%
    Spread: −0.05%, effectively no difference

But it does confirm strong risk separation:

    5% downside rate was 17.20 percentage points lower
    10% downside rate was 7.53 percentage points lower
    Average worst 20-day return was 1.86 percentage points better
    Outperform rate was 4.10 percentage points higher

So the combined classification currently works better as a risk-screening system than as an alpha-selection strategy.

Next, test whether these differences are stable across the same months rather than being driven by different class availability.

### Paired monthly spread and bootstrap confidence intervals

In [8]:
# Compare Attractive and Unfavourable classes on the same rebalance months

comparison_data = monthly_class_performance.loc[
    monthly_class_performance["opportunity_risk_class"].isin(
        [
            "Attractive Risk-Reward",
            "Unfavourable Risk-Reward",
        ]
    )
].copy()

comparison_data["evaluation_period"] = np.where(
    comparison_data["date"] < pd.Timestamp("2025-01-01"),
    "2024 Validation",
    "2025+ Test",
)


metrics_to_compare = {
    "average_future_return_20d": "Future return",
    "average_excess_return_20d": "Excess return",
    "average_future_min_return_20d": "Worst-path return",
    "outperform_rate": "Outperform rate",
    "downside_5pct_rate": "5% downside rate",
    "downside_10pct_rate": "10% downside rate",
}


def bootstrap_mean_ci(
    values,
    n_bootstrap=10_000,
    confidence_level=0.95,
    random_state=42,
):
    """Bootstrap confidence interval for the mean."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    rng = np.random.default_rng(random_state)

    bootstrap_means = np.empty(n_bootstrap)

    for index in range(n_bootstrap):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )
        bootstrap_means[index] = sample.mean()

    alpha = 1 - confidence_level

    lower = np.quantile(
        bootstrap_means,
        alpha / 2,
    )

    upper = np.quantile(
        bootstrap_means,
        1 - alpha / 2,
    )

    return lower, upper


paired_spread_rows = []

for period_name in ["2024 Validation", "2025+ Test"]:

    period_data = comparison_data.loc[
        comparison_data["evaluation_period"] == period_name
    ]

    attractive = (
        period_data.loc[
            period_data["opportunity_risk_class"]
            == "Attractive Risk-Reward"
        ]
        .set_index("date")
    )

    unfavourable = (
        period_data.loc[
            period_data["opportunity_risk_class"]
            == "Unfavourable Risk-Reward"
        ]
        .set_index("date")
    )

    common_dates = attractive.index.intersection(
        unfavourable.index
    )

    for metric, metric_label in metrics_to_compare.items():

        monthly_spread = (
            attractive.loc[common_dates, metric]
            - unfavourable.loc[common_dates, metric]
        )

        ci_lower, ci_upper = bootstrap_mean_ci(
            monthly_spread.values
        )

        paired_spread_rows.append(
            {
                "evaluation_period": period_name,
                "metric": metric_label,
                "paired_months": len(common_dates),
                "mean_spread_pct": monthly_spread.mean() * 100,
                "median_spread_pct": monthly_spread.median() * 100,
                "ci_95_lower_pct": ci_lower * 100,
                "ci_95_upper_pct": ci_upper * 100,
                "positive_spread_month_rate_pct": (
                    (monthly_spread > 0).mean() * 100
                ),
            }
        )


paired_monthly_spread_summary = (
    pd.DataFrame(paired_spread_rows)
    .round(2)
)

print("Paired monthly Attractive minus Unfavourable comparison:")
display(paired_monthly_spread_summary)

Paired monthly Attractive minus Unfavourable comparison:


,evaluation_period,metric,paired_months,mean_spread_pct,median_spread_pct,ci_95_lower_pct,ci_95_upper_pct,positive_spread_month_rate_pct
0,2024 Validation,Future return,11,3.56,2.62,0.59,6.81,72.73
1,2024 Validation,Excess return,11,3.56,2.62,0.59,6.81,72.73
2,2024 Validation,Worst-path return,11,2.96,2.13,1.06,5.02,81.82
3,2024 Validation,Outperform rate,11,10.37,1.75,-5.83,27.76,54.55
4,2024 Validation,5% downside rate,11,-26.52,-31.25,-37.90,-14.67,9.09
5,2024 Validation,10% downside rate,11,-18.92,-15.38,-30.35,-9.00,9.09
6,2025+ Test,Future return,18,-0.05,1.36,-2.18,1.84,61.11
7,2025+ Test,Excess return,18,-0.05,1.36,-2.18,1.84,61.11
8,2025+ Test,Worst-path return,18,1.86,1.06,0.85,2.93,83.33
9,2025+ Test,Outperform rate,18,4.10,2.78,-9.43,17.61,50.00


The paired test confirms the combined label is mainly useful for risk control, not return prediction.

For the cleaner 2025+ test period:

    Future-return spread: −0.05%, with the confidence interval crossing zero → no reliable alpha advantage.
    Worst-path return spread: +1.86%, with a fully positive confidence interval → Attractive stocks had meaningfully shallower drawdowns.
    5% downside-event spread: −17.20 percentage points, with a fully negative confidence interval → strong and stable risk reduction.
    10% downside-event spread: −7.53 points, but the interval slightly crosses zero → favourable, though not statistically stable yet.

Next, determine whether this separation comes mainly from the opportunity ranking or the downside-risk model.

### Evaluate opportunity tiers and risk bands separately

This will show whether the downside model is responsible for most of the historical separation and whether the outperform tiers add any independent return-ranking value.

In [9]:
# Decompose combined classifications into their two underlying signals

test_backtest = monthly_backtest.loc[
    monthly_backtest["date"] >= pd.Timestamp("2025-01-01")
].copy()


def create_signal_summary(
    frame: pd.DataFrame,
    group_column: str,
    category_order: list[str],
) -> pd.DataFrame:
    """Create equal-month-weighted outcome summary for one signal."""

    monthly_summary = (
        frame
        .groupby(
            ["date", group_column],
            observed=True,
        )
        .agg(
            stock_count=("yf_ticker", "size"),
            future_return=("future_return_20d", "mean"),
            excess_return=(
                "future_excess_return_20d_vs_nifty50",
                "mean",
            ),
            worst_path_return=("future_min_return_20d", "mean"),
            outperform_rate=(
                "target_outperform_nifty50_20d",
                "mean",
            ),
            downside_5pct_rate=(
                "target_downside_5pct_20d",
                "mean",
            ),
            downside_10pct_rate=(
                "target_big_downside_10pct_20d",
                "mean",
            ),
        )
        .reset_index()
    )

    summary = (
        monthly_summary
        .groupby(group_column, observed=True)
        .agg(
            months_present=("date", "nunique"),
            average_stocks_per_month=("stock_count", "mean"),
            average_future_return_20d=("future_return", "mean"),
            average_excess_return_20d=("excess_return", "mean"),
            average_future_min_return_20d=(
                "worst_path_return",
                "mean",
            ),
            outperform_rate=("outperform_rate", "mean"),
            downside_5pct_rate=("downside_5pct_rate", "mean"),
            downside_10pct_rate=("downside_10pct_rate", "mean"),
        )
        .reindex(category_order)
        .reset_index()
    )

    percentage_columns = [
        "average_future_return_20d",
        "average_excess_return_20d",
        "average_future_min_return_20d",
        "outperform_rate",
        "downside_5pct_rate",
        "downside_10pct_rate",
    ]

    summary[percentage_columns] = (
        summary[percentage_columns] * 100
    )

    return summary.round(2)


opportunity_tier_summary = create_signal_summary(
    frame=test_backtest,
    group_column="opportunity_tier",
    category_order=[
        "High Opportunity",
        "Moderate Opportunity",
        "Low Opportunity",
    ],
)

downside_band_summary = create_signal_summary(
    frame=test_backtest,
    group_column="downside_risk_band",
    category_order=[
        "Low Risk",
        "Watch Risk",
        "High Risk",
        "Very High Risk",
    ],
)


print("2025+ performance by opportunity tier:")
display(opportunity_tier_summary)

print("\n2025+ performance by downside-risk band:")
display(downside_band_summary)

2025+ performance by opportunity tier:


,opportunity_tier,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,High Opportunity,18,18.0,1.02,0.94,-4.00,55.56,31.79,11.11
1,Moderate Opportunity,18,27.0,1.39,1.30,-3.62,55.56,30.86,12.35
2,Low Opportunity,18,45.0,0.57,0.49,-4.44,51.23,35.06,14.07



2025+ performance by downside-risk band:


,downside_risk_band,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Low Risk,14,53.93,0.97,0.58,-3.25,54.81,23.41,8.62
1,Watch Risk,18,15.78,-0.48,-0.55,-4.64,43.54,34.39,16.12
2,High Risk,15,11.80,1.54,1.30,-4.27,61.63,32.44,15.32
3,Very High Risk,18,22.44,0.92,0.83,-5.16,49.4,48.64,15.62


The results show two things:

    Opportunity tiers: High and Moderate Opportunity both outperform Low Opportunity, but Moderate performs better than High. The signal has some ranking value, but it is not strongly monotonic.
    
    Downside bands: Low Risk is clearly safer, but Watch, High, and Very High are not cleanly ordered. This suggests the fixed probability thresholds may be unstable because the downside scores are not calibrated.

Next, test rank-based downside quintiles. This compares stocks relative to others on the same date and avoids depending on fixed probability cutoffs.

### Test downside-risk quintiles

In [10]:
# Evaluate downside risk using cross-sectional quintiles instead of fixed thresholds

test_backtest = monthly_backtest.loc[
    monthly_backtest["date"] >= pd.Timestamp("2025-01-01")
].copy()

test_backtest["downside_rank_percentile"] = (
    test_backtest["downside_risk_rank"]
    / test_backtest["rebalance_stock_count"]
)

test_backtest["downside_risk_quintile"] = pd.cut(
    test_backtest["downside_rank_percentile"],
    bins=[0.00, 0.20, 0.40, 0.60, 0.80, 1.00],
    labels=[
        "Q1 Highest Risk",
        "Q2 High Risk",
        "Q3 Medium Risk",
        "Q4 Lower Risk",
        "Q5 Lowest Risk",
    ],
    include_lowest=True,
)


monthly_quintile_performance = (
    test_backtest
    .groupby(
        ["date", "downside_risk_quintile"],
        observed=True,
    )
    .agg(
        stock_count=("yf_ticker", "size"),
        average_future_return_20d=("future_return_20d", "mean"),
        average_excess_return_20d=(
            "future_excess_return_20d_vs_nifty50",
            "mean",
        ),
        average_future_min_return_20d=(
            "future_min_return_20d",
            "mean",
        ),
        outperform_rate=(
            "target_outperform_nifty50_20d",
            "mean",
        ),
        downside_5pct_rate=(
            "target_downside_5pct_20d",
            "mean",
        ),
        downside_10pct_rate=(
            "target_big_downside_10pct_20d",
            "mean",
        ),
    )
    .reset_index()
)


quintile_order = [
    "Q1 Highest Risk",
    "Q2 High Risk",
    "Q3 Medium Risk",
    "Q4 Lower Risk",
    "Q5 Lowest Risk",
]

downside_quintile_summary = (
    monthly_quintile_performance
    .groupby(
        "downside_risk_quintile",
        observed=True,
    )
    .agg(
        months_present=("date", "nunique"),
        average_stocks_per_month=("stock_count", "mean"),
        average_future_return_20d=(
            "average_future_return_20d",
            "mean",
        ),
        average_excess_return_20d=(
            "average_excess_return_20d",
            "mean",
        ),
        average_future_min_return_20d=(
            "average_future_min_return_20d",
            "mean",
        ),
        outperform_rate=("outperform_rate", "mean"),
        downside_5pct_rate=("downside_5pct_rate", "mean"),
        downside_10pct_rate=("downside_10pct_rate", "mean"),
    )
    .reindex(quintile_order)
    .reset_index()
)


percentage_columns = [
    "average_future_return_20d",
    "average_excess_return_20d",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

downside_quintile_summary_display = downside_quintile_summary.copy()

downside_quintile_summary_display[percentage_columns] = (
    downside_quintile_summary_display[percentage_columns] * 100
)

downside_quintile_summary_display = (
    downside_quintile_summary_display.round(2)
)

print("2025+ downside-risk performance by cross-sectional quintile:")
display(downside_quintile_summary_display)

2025+ downside-risk performance by cross-sectional quintile:


,downside_risk_quintile,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Q1 Highest Risk,18,18.0,1.65,1.57,-4.96,54.94,40.74,18.52
1,Q2 High Risk,18,18.0,1.51,1.42,-4.22,55.56,35.49,12.35
2,Q3 Medium Risk,18,18.0,0.54,0.45,-4.35,50.31,34.57,14.2
3,Q4 Lower Risk,18,18.0,0.53,0.44,-3.50,54.63,27.78,9.57
4,Q5 Lowest Risk,18,18.0,0.32,0.24,-3.51,51.54,27.16,10.19


The quintile test is much cleaner than the fixed risk bands.

Key result for 2025+:

    Highest-risk quintile had a 40.74% 5% downside rate versus 27.16% for the lowest-risk quintile.
    Highest-risk quintile had an 18.52% 10% downside rate versus 10.19% for the lowest-risk quintile.
    Worst-path return was −4.96% versus −3.51%.
    High-risk stocks also produced higher returns, confirming the model measures drawdown exposure, not necessarily poor final returns.

The ordering is not perfect in every middle quintile, but the extreme groups show meaningful risk separation.

### Paired highest-risk versus lowest-risk test

In [11]:
# Compare highest-risk and lowest-risk quintiles on the same months

extreme_quintiles = monthly_quintile_performance.loc[
    monthly_quintile_performance["downside_risk_quintile"].isin(
        [
            "Q1 Highest Risk",
            "Q5 Lowest Risk",
        ]
    )
].copy()

highest_risk = (
    extreme_quintiles.loc[
        extreme_quintiles["downside_risk_quintile"]
        == "Q1 Highest Risk"
    ]
    .set_index("date")
)

lowest_risk = (
    extreme_quintiles.loc[
        extreme_quintiles["downside_risk_quintile"]
        == "Q5 Lowest Risk"
    ]
    .set_index("date")
)

common_dates = highest_risk.index.intersection(
    lowest_risk.index
)

risk_metrics = {
    "average_future_return_20d": "Future return",
    "average_excess_return_20d": "Excess return",
    "average_future_min_return_20d": "Worst-path return",
    "downside_5pct_rate": "5% downside rate",
    "downside_10pct_rate": "10% downside rate",
}

risk_spread_rows = []

for metric, metric_label in risk_metrics.items():

    # Highest Risk minus Lowest Risk
    monthly_spread = (
        highest_risk.loc[common_dates, metric]
        - lowest_risk.loc[common_dates, metric]
    )

    ci_lower, ci_upper = bootstrap_mean_ci(
        monthly_spread.values
    )

    risk_spread_rows.append(
        {
            "metric": metric_label,
            "paired_months": len(common_dates),
            "mean_high_minus_low_pct": (
                monthly_spread.mean() * 100
            ),
            "median_high_minus_low_pct": (
                monthly_spread.median() * 100
            ),
            "ci_95_lower_pct": ci_lower * 100,
            "ci_95_upper_pct": ci_upper * 100,
            "positive_spread_month_rate_pct": (
                (monthly_spread > 0).mean() * 100
            ),
        }
    )

highest_vs_lowest_risk_summary = (
    pd.DataFrame(risk_spread_rows)
    .round(2)
)

print("2025+ Highest-Risk minus Lowest-Risk comparison:")
display(highest_vs_lowest_risk_summary)

2025+ Highest-Risk minus Lowest-Risk comparison:


,metric,paired_months,mean_high_minus_low_pct,median_high_minus_low_pct,ci_95_lower_pct,ci_95_upper_pct,positive_spread_month_rate_pct
0,Future return,18,1.33,0.63,-0.73,3.46,55.56
1,Excess return,18,1.34,0.63,-0.72,3.47,55.56
2,Worst-path return,18,-1.45,-1.84,-2.55,-0.39,27.78
3,5% downside rate,18,13.58,11.11,6.17,22.53,72.22
4,10% downside rate,18,8.33,5.56,1.54,16.67,55.56


This is strong evidence that the downside model works as a relative risk-ranking model.

For 2025+:

    Highest-risk stocks experienced 1.45 percentage points deeper drawdowns.
    Their 5% downside-event rate was 13.58 points higher.
    Their 10% downside-event rate was 8.33 points higher.
    All three confidence intervals exclude zero, so the risk separation is statistically meaningful.
    The return spread is not reliable, confirming that high downside risk does not necessarily mean low final return.

Next, test the outperform model using equal-sized quintiles.

### Test highest versus lowest opportunity quintiles

For return, excess return, and outperform rate, a positive spread favours the opportunity model.

In [12]:
# Evaluate the outperform ranking using equal-sized cross-sectional quintiles

opportunity_test = monthly_backtest.loc[
    monthly_backtest["date"] >= pd.Timestamp("2025-01-01")
].copy()

opportunity_test["opportunity_quintile"] = pd.cut(
    opportunity_test["outperform_rank_percentile"],
    bins=[0.00, 0.20, 0.40, 0.60, 0.80, 1.00],
    labels=[
        "Q1 Highest Opportunity",
        "Q2 High Opportunity",
        "Q3 Medium Opportunity",
        "Q4 Lower Opportunity",
        "Q5 Lowest Opportunity",
    ],
    include_lowest=True,
)


monthly_opportunity_performance = (
    opportunity_test
    .groupby(
        ["date", "opportunity_quintile"],
        observed=True,
    )
    .agg(
        stock_count=("yf_ticker", "size"),
        average_future_return_20d=("future_return_20d", "mean"),
        average_excess_return_20d=(
            "future_excess_return_20d_vs_nifty50",
            "mean",
        ),
        average_future_min_return_20d=(
            "future_min_return_20d",
            "mean",
        ),
        outperform_rate=(
            "target_outperform_nifty50_20d",
            "mean",
        ),
        downside_5pct_rate=(
            "target_downside_5pct_20d",
            "mean",
        ),
        downside_10pct_rate=(
            "target_big_downside_10pct_20d",
            "mean",
        ),
    )
    .reset_index()
)


opportunity_quintile_order = [
    "Q1 Highest Opportunity",
    "Q2 High Opportunity",
    "Q3 Medium Opportunity",
    "Q4 Lower Opportunity",
    "Q5 Lowest Opportunity",
]

opportunity_quintile_summary = (
    monthly_opportunity_performance
    .groupby(
        "opportunity_quintile",
        observed=True,
    )
    .agg(
        months_present=("date", "nunique"),
        average_stocks_per_month=("stock_count", "mean"),
        average_future_return_20d=(
            "average_future_return_20d",
            "mean",
        ),
        average_excess_return_20d=(
            "average_excess_return_20d",
            "mean",
        ),
        average_future_min_return_20d=(
            "average_future_min_return_20d",
            "mean",
        ),
        outperform_rate=("outperform_rate", "mean"),
        downside_5pct_rate=("downside_5pct_rate", "mean"),
        downside_10pct_rate=("downside_10pct_rate", "mean"),
    )
    .reindex(opportunity_quintile_order)
    .reset_index()
)


percentage_columns = [
    "average_future_return_20d",
    "average_excess_return_20d",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

opportunity_quintile_summary_display = (
    opportunity_quintile_summary.copy()
)

opportunity_quintile_summary_display[percentage_columns] = (
    opportunity_quintile_summary_display[percentage_columns] * 100
)

opportunity_quintile_summary_display = (
    opportunity_quintile_summary_display.round(2)
)


# Paired highest-opportunity versus lowest-opportunity comparison
highest_opportunity = (
    monthly_opportunity_performance.loc[
        monthly_opportunity_performance["opportunity_quintile"]
        == "Q1 Highest Opportunity"
    ]
    .set_index("date")
)

lowest_opportunity = (
    monthly_opportunity_performance.loc[
        monthly_opportunity_performance["opportunity_quintile"]
        == "Q5 Lowest Opportunity"
    ]
    .set_index("date")
)

common_dates = highest_opportunity.index.intersection(
    lowest_opportunity.index
)

opportunity_metrics = {
    "average_future_return_20d": "Future return",
    "average_excess_return_20d": "Excess return",
    "average_future_min_return_20d": "Worst-path return",
    "outperform_rate": "Outperform rate",
    "downside_5pct_rate": "5% downside rate",
    "downside_10pct_rate": "10% downside rate",
}

opportunity_spread_rows = []

for metric, metric_label in opportunity_metrics.items():

    monthly_spread = (
        highest_opportunity.loc[common_dates, metric]
        - lowest_opportunity.loc[common_dates, metric]
    )

    ci_lower, ci_upper = bootstrap_mean_ci(
        monthly_spread.values
    )

    opportunity_spread_rows.append(
        {
            "metric": metric_label,
            "paired_months": len(common_dates),
            "mean_high_minus_low_pct": monthly_spread.mean() * 100,
            "median_high_minus_low_pct": monthly_spread.median() * 100,
            "ci_95_lower_pct": ci_lower * 100,
            "ci_95_upper_pct": ci_upper * 100,
            "positive_spread_month_rate_pct": (
                (monthly_spread > 0).mean() * 100
            ),
        }
    )

highest_vs_lowest_opportunity_summary = (
    pd.DataFrame(opportunity_spread_rows)
    .round(2)
)


print("2025+ performance by opportunity quintile:")
display(opportunity_quintile_summary_display)

print("\nHighest-Opportunity minus Lowest-Opportunity comparison:")
display(highest_vs_lowest_opportunity_summary)

2025+ performance by opportunity quintile:


,opportunity_quintile,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Q1 Highest Opportunity,18,18.0,1.02,0.94,-4.00,55.56,31.79,11.11
1,Q2 High Opportunity,18,18.0,1.42,1.34,-3.63,56.17,29.94,12.65
2,Q3 Medium Opportunity,18,18.0,1.34,1.26,-3.65,54.63,30.56,10.19
3,Q4 Lower Opportunity,18,18.0,0.80,0.72,-4.32,53.7,34.88,12.65
4,Q5 Lowest Opportunity,18,18.0,-0.05,-0.13,-4.92,46.91,38.58,18.21



Highest-Opportunity minus Lowest-Opportunity comparison:


,metric,paired_months,mean_high_minus_low_pct,median_high_minus_low_pct,ci_95_lower_pct,ci_95_upper_pct,positive_spread_month_rate_pct
0,Future return,18,1.07,1.16,-0.12,2.36,66.67
1,Excess return,18,1.07,1.16,-0.13,2.36,66.67
2,Worst-path return,18,0.92,0.54,0.04,1.91,61.11
3,Outperform rate,18,8.64,5.56,1.85,15.43,72.22
4,5% downside rate,18,-6.79,-5.56,-14.20,0.62,27.78
5,10% downside rate,18,-7.10,-5.56,-11.73,-2.78,11.11


The opportunity model shows useful ranking separation, though not yet conclusive alpha:

        Highest-opportunity quintile earned 1.07 percentage points more than the lowest.
        The return confidence interval slightly crosses zero, so the return advantage is not statistically stable yet.
        Outperform rate improved by 8.64 points, with a confidence interval above zero.
        Severe 10% downside events were 7.10 points lower, also statistically meaningful.
        Worst-path returns were 0.92 points better.

This supports using the model for relative screening, not precise return forecasting.

Because downside ranks performed more consistently than fixed probability bands, the next step is to test a fully rank-based combined classification.

### Backtest rank-based combined classes

This will tell us whether Notebook 07 should replace the fixed downside thresholds with a more stable relative risk ranking.

In [13]:
# Create a rank-based combined classification for the 2025+ test period

rank_combined_test = monthly_backtest.loc[
    monthly_backtest["date"] >= pd.Timestamp("2025-01-01")
].copy()


# Top half of downside ranking = higher relative risk
rank_combined_test["relative_risk_group"] = np.where(
    (
        rank_combined_test["downside_risk_rank"]
        / rank_combined_test["rebalance_stock_count"]
    ) <= 0.50,
    "Higher Relative Risk",
    "Lower Relative Risk",
)


def assign_rank_based_combined_class(row) -> str:
    opportunity = row["opportunity_tier"]
    risk = row["relative_risk_group"]

    lower_risk = risk == "Lower Relative Risk"

    if opportunity == "High Opportunity" and lower_risk:
        return "Attractive Risk-Reward"

    if opportunity == "High Opportunity":
        return "High Opportunity / High Risk"

    if opportunity == "Moderate Opportunity" and lower_risk:
        return "Balanced Opportunity"

    if opportunity == "Moderate Opportunity":
        return "Caution"

    if opportunity == "Low Opportunity" and lower_risk:
        return "Low Opportunity / Lower Risk"

    return "Unfavourable Risk-Reward"


rank_combined_test["rank_based_combined_class"] = (
    rank_combined_test.apply(
        assign_rank_based_combined_class,
        axis=1,
    )
)


# Equal-weight performance for every class and month
monthly_rank_class_performance = (
    rank_combined_test
    .groupby(
        ["date", "rank_based_combined_class"],
        observed=True,
    )
    .agg(
        stock_count=("yf_ticker", "size"),
        future_return=("future_return_20d", "mean"),
        excess_return=(
            "future_excess_return_20d_vs_nifty50",
            "mean",
        ),
        worst_path_return=("future_min_return_20d", "mean"),
        outperform_rate=(
            "target_outperform_nifty50_20d",
            "mean",
        ),
        downside_5pct_rate=(
            "target_downside_5pct_20d",
            "mean",
        ),
        downside_10pct_rate=(
            "target_big_downside_10pct_20d",
            "mean",
        ),
    )
    .reset_index()
)


rank_class_order = [
    "Attractive Risk-Reward",
    "High Opportunity / High Risk",
    "Balanced Opportunity",
    "Caution",
    "Low Opportunity / Lower Risk",
    "Unfavourable Risk-Reward",
]


rank_class_summary = (
    monthly_rank_class_performance
    .groupby(
        "rank_based_combined_class",
        observed=True,
    )
    .agg(
        months_present=("date", "nunique"),
        average_stocks_per_month=("stock_count", "mean"),
        average_future_return_20d=("future_return", "mean"),
        average_excess_return_20d=("excess_return", "mean"),
        average_future_min_return_20d=(
            "worst_path_return",
            "mean",
        ),
        outperform_rate=("outperform_rate", "mean"),
        downside_5pct_rate=("downside_5pct_rate", "mean"),
        downside_10pct_rate=("downside_10pct_rate", "mean"),
    )
    .reindex(rank_class_order)
    .reset_index()
)


percentage_columns = [
    "average_future_return_20d",
    "average_excess_return_20d",
    "average_future_min_return_20d",
    "outperform_rate",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

rank_class_summary_display = rank_class_summary.copy()

rank_class_summary_display[percentage_columns] = (
    rank_class_summary_display[percentage_columns] * 100
)

rank_class_summary_display = rank_class_summary_display.round(2)


print("2025+ rank-based combined classification performance:")
display(rank_class_summary_display)

print("\nHistorical observation counts:")
display(
    rank_combined_test["rank_based_combined_class"]
    .value_counts()
    .rename_axis("rank_based_combined_class")
    .reset_index(name="observation_count")
)

2025+ rank-based combined classification performance:


,rank_based_combined_class,months_present,average_stocks_per_month,average_future_return_20d,average_excess_return_20d,average_future_min_return_20d,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Attractive Risk-Reward,18,10.39,0.88,0.79,-3.04,54.43,25.02,6.45
1,High Opportunity / High Risk,18,7.61,1.22,1.14,-4.98,56.49,44.91,15.13
2,Balanced Opportunity,18,15.83,0.48,0.40,-3.52,52.39,28.37,10.94
3,Caution,18,11.17,2.26,2.17,-3.74,58.07,34.53,13.94
4,Low Opportunity / Lower Risk,18,18.78,0.60,0.51,-3.74,54.41,28.98,11.91
5,Unfavourable Risk-Reward,18,26.22,0.90,0.81,-4.88,50.7,38.72,16.05



Historical observation counts:


,rank_based_combined_class,observation_count
0,Unfavourable Risk-Reward,472
1,Low Opportunity / Lower Risk,338
2,Balanced Opportunity,285
3,Caution,201
4,Attractive Risk-Reward,187
5,High Opportunity / High Risk,137


The rank-based classification gives cleaner risk separation:

        Attractive Risk-Reward had the shallowest average drawdown: −3.04%
        Its 10% downside-event rate was 6.45%
        Unfavourable Risk-Reward had a deeper drawdown: −4.88%
        Its 10% downside-event rate was 16.05%

However, average excess returns were almost identical: 0.79% versus 0.81%. So the classification is still mainly a risk-screening framework, not a proven return strategy.

Before changing Notebook 07, run a paired statistical comparison.

### Validate rank-based extreme classes

In [14]:
# Paired monthly comparison:
# Attractive Risk-Reward minus Unfavourable Risk-Reward

rank_attractive = (
    monthly_rank_class_performance.loc[
        monthly_rank_class_performance["rank_based_combined_class"]
        == "Attractive Risk-Reward"
    ]
    .set_index("date")
)

rank_unfavourable = (
    monthly_rank_class_performance.loc[
        monthly_rank_class_performance["rank_based_combined_class"]
        == "Unfavourable Risk-Reward"
    ]
    .set_index("date")
)

common_dates = rank_attractive.index.intersection(
    rank_unfavourable.index
)

rank_combined_metrics = {
    "future_return": "Future return",
    "excess_return": "Excess return",
    "worst_path_return": "Worst-path return",
    "outperform_rate": "Outperform rate",
    "downside_5pct_rate": "5% downside rate",
    "downside_10pct_rate": "10% downside rate",
}

rank_combined_spread_rows = []

for metric, metric_label in rank_combined_metrics.items():

    monthly_spread = (
        rank_attractive.loc[common_dates, metric]
        - rank_unfavourable.loc[common_dates, metric]
    )

    ci_lower, ci_upper = bootstrap_mean_ci(
        monthly_spread.values
    )

    rank_combined_spread_rows.append(
        {
            "metric": metric_label,
            "paired_months": len(common_dates),
            "mean_attractive_minus_unfavourable_pct": (
                monthly_spread.mean() * 100
            ),
            "median_spread_pct": (
                monthly_spread.median() * 100
            ),
            "ci_95_lower_pct": ci_lower * 100,
            "ci_95_upper_pct": ci_upper * 100,
            "positive_spread_month_rate_pct": (
                (monthly_spread > 0).mean() * 100
            ),
        }
    )

rank_combined_paired_summary = (
    pd.DataFrame(rank_combined_spread_rows)
    .round(2)
)

print(
    "2025+ Rank-Based Attractive minus "
    "Unfavourable comparison:"
)

display(rank_combined_paired_summary)

"""
For interpretation:

Positive return and worst-path spreads favour Attractive.
Negative downside-event spreads favour Attractive.
Confidence intervals excluding zero indicate stable historical separation.
"""

2025+ Rank-Based Attractive minus Unfavourable comparison:


,metric,paired_months,mean_attractive_minus_unfavourable_pct,median_spread_pct,ci_95_lower_pct,ci_95_upper_pct,positive_spread_month_rate_pct
0,Future return,18,-0.02,0.54,-2.25,1.99,55.56
1,Excess return,18,-0.02,0.54,-2.26,1.99,55.56
2,Worst-path return,18,1.84,1.23,0.73,3.14,88.89
3,Outperform rate,18,3.73,-1.34,-10.33,17.83,44.44
4,5% downside rate,18,-13.70,-6.74,-24.59,-3.98,22.22
5,10% downside rate,18,-9.60,-4.85,-19.70,-1.99,11.11


'\nFor interpretation:\n\nPositive return and worst-path spreads favour Attractive.\nNegative downside-event spreads favour Attractive.\nConfidence intervals excluding zero indicate stable historical separation.\n'

The result validates the rank-based combined classification as a risk-screening system:

        No reliable return advantage: −0.02% spread, confidence interval crosses zero.
        Shallower drawdowns: +1.84 percentage points, statistically meaningful.
        5% downside events: 13.70 points lower, statistically meaningful.
        10% downside events: 9.60 points lower, statistically meaningful.


So we should use relative downside rank for the combined classification. The fixed probability bands can remain as descriptive alerts, but should not drive the main category.

### Save backtest results and final report

In [15]:
from datetime import datetime
import json


# Summary tables to save
tables_to_save = {
    "combined_class_summary_all_periods_v1.csv":
        class_backtest_summary_display,

    "combined_class_summary_by_period_v1.csv":
        period_class_summary_display,

    "fixed_class_paired_spread_v1.csv":
        paired_monthly_spread_summary,

    "downside_quintile_summary_2025plus_v1.csv":
        downside_quintile_summary_display,

    "downside_extreme_spread_2025plus_v1.csv":
        highest_vs_lowest_risk_summary,

    "opportunity_quintile_summary_2025plus_v1.csv":
        opportunity_quintile_summary_display,

    "opportunity_extreme_spread_2025plus_v1.csv":
        highest_vs_lowest_opportunity_summary,

    "rank_based_class_summary_2025plus_v1.csv":
        rank_class_summary_display,

    "rank_based_class_paired_spread_2025plus_v1.csv":
        rank_combined_paired_summary,
}


saved_report_paths = []

for filename, table in tables_to_save.items():
    output_path = BACKTEST_REPORT_DIR / filename

    table.to_csv(
        output_path,
        index=False,
    )

    saved_report_paths.append(output_path)


# Final decision metadata
backtest_metadata = {
    "backtest_version": "v1",
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "full_backtest_period": {
        "start": str(monthly_backtest["date"].min().date()),
        "end": str(monthly_backtest["date"].max().date()),
        "rebalance_frequency": "monthly",
        "rebalance_dates": int(monthly_backtest["date"].nunique()),
        "stocks_per_rebalance": 90,
        "observations": int(len(monthly_backtest)),
    },
    "clean_test_period": {
        "start": "2025-01-01",
        "months": 18,
    },
    "final_interpretation": {
        "opportunity_model": (
            "Useful as a relative ranking signal, but no statistically "
            "reliable return-alpha spread was established."
        ),
        "downside_model": (
            "Validated as a relative risk-ranking model with meaningful "
            "separation in drawdowns and downside-event rates."
        ),
        "combined_classification": (
            "Validated primarily for risk screening, not as a proven "
            "return-generating strategy."
        ),
    },
    "recommended_design": {
        "combined_classification_risk_input": (
            "Cross-sectional downside rank rather than fixed probability bands."
        ),
        "fixed_probability_bands": (
            "Retain as descriptive alerts, but do not use as the primary "
            "combined-classification rule."
        ),
    },
    "rank_based_attractive_minus_unfavourable_2025plus": {
        "future_return_spread_pct": -0.02,
        "future_return_ci_95_pct": [-2.25, 1.99],
        "worst_path_return_spread_pct": 1.84,
        "worst_path_return_ci_95_pct": [0.73, 3.14],
        "downside_5pct_rate_spread_pct": -13.70,
        "downside_5pct_rate_ci_95_pct": [-24.59, -3.98],
        "downside_10pct_rate_spread_pct": -9.60,
        "downside_10pct_rate_ci_95_pct": [-19.70, -1.99],
    },
}

BACKTEST_METADATA_PATH = (
    BACKTEST_REPORT_DIR
    / "historical_signal_backtest_metadata_v1.json"
)

with open(
    BACKTEST_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        backtest_metadata,
        file,
        indent=4,
    )


# Human-readable report
BACKTEST_SUMMARY_PATH = (
    BACKTEST_REPORT_DIR
    / "historical_signal_backtest_summary_v1.md"
)

summary_markdown = """# Historical Signal Backtest Summary

## Backtest Design

- Monthly rebalancing
- 90 prediction-ready stocks per rebalance
- 30 rebalance dates from 2024-01-02 through 2026-06-01
- 2,700 historical stock observations
- 2025 onward treated as the cleaner test period

## Main Findings

### Opportunity Model

The opportunity model provides useful relative ranking, but the return advantage
was not statistically conclusive.

In the 2025+ test period, the highest-opportunity quintile exceeded the
lowest-opportunity quintile by 1.07 percentage points, but the 95% confidence
interval crossed zero.

The highest-opportunity quintile did show:

- An 8.64 percentage-point higher outperform rate
- A 7.10 percentage-point lower 10% downside-event rate
- Shallower average drawdowns

The model should therefore be used as a ranking signal rather than as a precise
return forecast.

### Downside Model

The downside model successfully separated relative drawdown risk.

Compared with the lowest-risk quintile, the highest-risk quintile experienced:

- 1.45 percentage points deeper average drawdowns
- A 13.58 percentage-point higher 5% downside-event rate
- An 8.33 percentage-point higher 10% downside-event rate

All three differences had confidence intervals excluding zero.

### Combined Classification

The rank-based combined classification produced meaningful risk separation.

Compared with Unfavourable Risk-Reward stocks, Attractive Risk-Reward stocks had:

- No reliable return advantage
- 1.84 percentage points shallower average drawdowns
- A 13.70 percentage-point lower 5% downside-event rate
- A 9.60 percentage-point lower 10% downside-event rate

## Final Decision

The combined MarketGuard classification is validated as a risk-screening
framework, not as a proven alpha-generating strategy.

Relative downside ranking should be used for the primary combined
classification. Fixed probability bands may remain as descriptive alerts.

The dashboard should clearly present:

- Opportunity rank
- Relative downside-risk rank
- Combined risk-screening category
- Data confidence
- Raw model scores with a calibration warning
"""

BACKTEST_SUMMARY_PATH.write_text(
    summary_markdown,
    encoding="utf-8",
)


print("Saved summary tables:")

for path in saved_report_paths:
    print(path)

print("\nSaved metadata:")
print(BACKTEST_METADATA_PATH)

print("\nSaved Markdown report:")
print(BACKTEST_SUMMARY_PATH)

Saved summary tables:
E:\Projects\marketguard-india\reports\historical_signal_backtest\combined_class_summary_all_periods_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\combined_class_summary_by_period_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\fixed_class_paired_spread_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\downside_quintile_summary_2025plus_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\downside_extreme_spread_2025plus_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\opportunity_quintile_summary_2025plus_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\opportunity_extreme_spread_2025plus_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\rank_based_class_summary_2025plus_v1.csv
E:\Projects\marketguard-india\reports\historical_signal_backtest\rank_based_class_paired_spread_2025plus_v1.csv

Saved metadata:
E:\

### Validate saved backtest artifacts

In [16]:
# Validate saved backtest outputs

required_report_paths = [
    BACKTEST_REPORT_DIR / filename
    for filename in tables_to_save
]

required_report_paths.extend(
    [
        BACKTEST_METADATA_PATH,
        BACKTEST_SUMMARY_PATH,
    ]
)

missing_report_paths = [
    path
    for path in required_report_paths
    if not path.exists()
]

if missing_report_paths:
    missing_text = "\n".join(str(path) for path in missing_report_paths)

    raise FileNotFoundError(
        f"Missing saved backtest artifacts:\n{missing_text}"
    )


# Reload important result tables
validated_downside_spread = pd.read_csv(
    BACKTEST_REPORT_DIR
    / "downside_extreme_spread_2025plus_v1.csv"
)

validated_opportunity_spread = pd.read_csv(
    BACKTEST_REPORT_DIR
    / "opportunity_extreme_spread_2025plus_v1.csv"
)

validated_rank_combined_spread = pd.read_csv(
    BACKTEST_REPORT_DIR
    / "rank_based_class_paired_spread_2025plus_v1.csv"
)


# Confirm expected metrics exist
expected_metrics = {
    "Future return",
    "Excess return",
    "Worst-path return",
    "5% downside rate",
    "10% downside rate",
}

for table_name, table in [
    ("downside spread", validated_downside_spread),
    ("rank combined spread", validated_rank_combined_spread),
]:
    available_metrics = set(table["metric"])

    missing_metrics = expected_metrics - available_metrics

    if missing_metrics:
        raise ValueError(
            f"{table_name} is missing metrics: {missing_metrics}"
        )


# Confirm metadata and Markdown report are not empty
if BACKTEST_METADATA_PATH.stat().st_size == 0:
    raise ValueError("Backtest metadata file is empty.")

if BACKTEST_SUMMARY_PATH.stat().st_size == 0:
    raise ValueError("Backtest Markdown summary is empty.")


print("Historical backtest artifact validation passed.")

print("\nSaved artifact count:")
print(len(required_report_paths))

print("\nDownside spread rows:")
print(len(validated_downside_spread))

print("\nOpportunity spread rows:")
print(len(validated_opportunity_spread))

print("\nRank-based combined spread rows:")
print(len(validated_rank_combined_spread))

print("\nMetadata size:")
print(BACKTEST_METADATA_PATH.stat().st_size, "bytes")

print("\nMarkdown report size:")
print(BACKTEST_SUMMARY_PATH.stat().st_size, "bytes")

print("\nAll required reports present:")
print(True)

Historical backtest artifact validation passed.

Saved artifact count:
11

Downside spread rows:
5

Opportunity spread rows:
6

Rank-based combined spread rows:
6

Metadata size:
1755 bytes

Markdown report size:
2203 bytes

All required reports present:
True
